# V4: Regularni izrazi i udaljenost uređivanja

**I051 Računalno jezikoslovlje** - vježbe uz _3. i 4. predavanje_.

| | Predavanje | Vježbe |
|---|---|---|
| 1 | P3: osnovni pojmovi i operatori | `re.findall` u rečenici i u rječniku korpusa |
| 2 | P3: grupe i zamjena | datumi i iznosi u vijestima |
| 3 | P4: tokenizacija pravilima | tokenizator za hrvatski, segmentacija rečenica |
| 4 | P4: udaljenost uređivanja | `nltk.edit_distance`, ispravljanje pogrešaka |

# 0. Priprema

Kao u V2: u VSCode-u otvorite mapu s bilježnicom i odaberite Jupyter kernel. Paket `nltk` je popisan u `requirements.txt`; ćeliju je dovoljno pokrenuti jednom.

In [ ]:
%pip install -r requirements.txt

U mapi `data` je datoteka `hr_train.tsv`: rečenice hrvatskih vijesti iz korpusa UD Croatian SET (isti korpus kao u V1 i V2). Svaki redak ima tri stupca odvojena tabulatorom:

1. oznaku dokumenta (vijesti iz kojeg je rečenica),
2. rečenicu u **izvornom obliku**,
3. **zlatni standard**: tokene te rečenice, kako su ih odredili lingvisti, odvojene razmakom.

Učitajmo je u listu trojki `(dokument, rečenica, tokeni)`:

In [ ]:
import re

rows = []
for line in open("data/hr_train.tsv", encoding="utf-8"):
    doc_id, sentence, gold = line.rstrip("\n").split("\t")   # tri stupca
    rows.append((doc_id, sentence, gold.split(" ")))          # tokeni kao lista

print(len(rows), "rečenica")
doc_id, sentence, gold = rows[2]   # treća rečenica
print(doc_id)
print(sentence)
print(gold)

# 1. Regularni izrazi: ponavljanje

S predavanja P3: `re.findall(uzorak, tekst)` vraća listu svih podudaranja. Uzorke pišemo kao sirove stringove `r"…"`.

| zapis | značenje |
|---|---|
| `[čć]`, `[a-z]`, `[^aeiou]` | jedan znak iz skupa, raspon, negacija |
| `?`, `*`, `+`, `{n}` | 0–1, 0 ili više, 1 ili više, točno n puta |
| `.` | bilo koji znak |
| `^`, `$`, `\b` | početak, kraj, granica riječi |
| `\d`, `\w`, `\s` | znamenka, riječni znak, praznina |

In [ ]:
sentence = rows[2][1]
print(sentence)
print(re.findall(r"\d+", sentence))         # brojevi
print(re.findall(r"\b[A-Z]\w+", sentence))  # riječi s velikim početnim slovom

Operatore je najlakše isprobati na **rječniku** korpusa: listi svih različitih riječi (malim slovima, samo od slova). Funkcija `find` vraća riječi rječnika u kojima postoji podudaranje.

In [ ]:
vocab = sorted(set(t.lower() for _, _, gold in rows for t in gold if t.isalpha()))
print(len(vocab), "različnica")

def find(pattern):
    return [w for w in vocab if re.search(pattern, w)]

print(find(r"^ma[čć]"))       # počinje s mač ili mać
print(find(r"^k.ć.$"))        # križaljka: k _ ć _
print(find(r"^hrvatsk"))      # počinje s hrvatsk

### Zadatak 1

Napišite izraze i ispišite koliko je riječi pronađeno i prvih deset:

1. riječi koje završavaju na *-ost* (budućnost, djelatnost …),
2. riječi od točno tri slova koje počinju s *p*,
3. riječi koje počinju s *naj* i završavaju na *iji* (najnoviji …).

In [ ]:
for pattern in [r"...", r"...", r"..."]:   # dopunite
    words = find(pattern)
    print(pattern, len(words), words[:10])

# 2. Grupe i zamjena

Spojimo sve rečenice u jedan tekst vijesti.

**Grupe za hvatanje** `( )` pamte dijelove podudaranja; tada `re.findall` vraća n-torke. Datum u vijestima često izgleda ovako: *22. lipnja 2007.*

In [ ]:
text = "\n".join(sentence for _, sentence, _ in rows)

dates = re.findall(r"\b(\d{1,2})\. ([a-zčćđšž]+) (\d{4})\.", text)   # dan, mjesec, godina
print(len(dates))
dates[:5]

U zamjeni `re.sub(uzorak, zamjena, tekst)` grupe dohvaćamo s `\1`, `\2`, `\3`:

In [ ]:
print(re.sub(r"\b(\d{1,2})\. ([a-zčćđšž]+) (\d{4})\.", r"\3., \1. \2", "Potpisan je 22. lipnja 2007. u Bruxellesu."))
print(re.sub(r"\d", "#", "OIB 12345678901"))   # skrivanje znamenki

**Pogled unaprijed** `(?= … )` provjerava što slijedi, a ne uzima to u rezultat. Iznosi u eurima:

In [ ]:
euros = re.findall(r"\d+(?:[.,]\d+)*(?= eura)", text)   # broj iza kojeg slijedi » eura«
print(len(euros), euros[:10])

**Pokušajte:** pronađite iznose u **dolarima**.

In [ ]:
# vaš kod

# 3. Tokenizacija pravilima

S predavanja P4: tokenizator pravilima je regularni izraz koji opisuje **jedan token**; `re.findall` vrati sve tokene rečenice. Zlatni standard nam govori je li rezultat točan.

Funkcija `accuracy` mjeri **udio rečenica** u kojima je tokenizator dao točno iste tokene kao zlatni standard. Isprobajmo tri jednostavna tokenizatora:

* `\S+` — sve između razmaka (isto kao `split()`),
* `\w+` — samo riječi, bez interpunkcije,
* `\w+|[^\w\s]` — riječ ili jedan znak interpunkcije.

In [ ]:
def accuracy(pattern):
    correct = 0
    for _, sentence, gold in rows:
        if re.findall(pattern, sentence) == gold:
            correct += 1
    return correct / len(rows)

for pattern in [r"\S+", r"\w+", r"\w+|[^\w\s]"]:
    print(f"{pattern:12} {accuracy(pattern):.1%}")

`[^\w\s]` je jedan znak koji nije ni riječni znak ni praznina — dakle interpunkcija. Pogledajmo gdje treći tokenizator griješi: ispisujemo tokene kojih nema u zlatnom standardu.

In [ ]:
def show_errors(pattern, n=5):
    for _, sentence, gold in rows:
        tokens = re.findall(pattern, sentence)
        if tokens != gold:
            print([t for t in tokens if t not in gold], "→", [t for t in gold if t not in tokens])
            n -= 1
            if n == 0:
                break

show_errors(r"\w+|[^\w\s]")

### Zadatak 2

Duge izraze pišemo u više redaka: `(?x)` na početku dopušta razmake i komentare `#`. Alternative se isprobavaju **redom**, pa posebna pravila idu **iznad** općih.

Dodajte pravila jedno po jedno i nakon svakog pokrenite ćeliju:

1. brojevi s točkom ili zarezom: `10.000`, `4,5` — `\d+(?:[.,]\d+)+`
2. kratice: `dr.`, `npr.`, `tj.`, `tzv.`, … — `(?:dr|npr|tj|tzv|sv|mr|prof)\.`
3. redni brojevi: točka pripada broju ako iza nje slijedi razmak i malo slovo, npr. *21. svibnja* — `\d+\.(?=\s[a-zčćđšž])`
4. trotočka i dvostruka crtica: `...` i `--` — `\.\.\.|--`

Koliko vam je točnost na kraju?

In [ ]:
pattern = r'''(?x)
    \w+(?:-\w+)*    # riječi, i s crticom: NATO-a
  | [^\w\s]         # interpunkcija
'''
# dopunite: nova pravila stavite iznad postojećih

print(f"{accuracy(pattern):.1%}")
show_errors(pattern)

U načinu `(?x)` razmaci u uzorku se zanemaruju, pa razmak u tekstu tražimo s `\s`.

**Za razmisliti:** je li *2008.* na kraju rečenice redni broj ili broj iza kojeg je kraj rečenice?

## 3.1 Segmentacija rečenica

Spojimo rečenice svakog dokumenta natrag u tekst i pokušajmo ih ponovno razdvojiti. Dijelimo na praznini kojoj **prethodi** `.`, `!` ili `?` (pogled unatrag `(?<= … )`) i iza koje **slijedi** veliko slovo (pogled unaprijed).

In [ ]:
docs = {}
for doc_id, sentence, _ in rows:
    docs.setdefault(doc_id, []).append(sentence)   # rečenice po dokumentima

def split_sentences(text):
    return re.split(r"(?<=[.!?])\s+(?=[A-ZČĆĐŠŽ])", text)

example = "Sastanak je 5. listopada u 10 h. Dr. Horvat kasni. Dođite na vrijeme!"
split_sentences(example)

Kao na predavanju P3 mjerimo **preciznost** (udio pronađenih rečenica koje su točne) i **odziv** (udio pravih rečenica koje smo pronašli):

In [ ]:
def evaluate(split):
    found = correct = total = 0
    for sentences in docs.values():
        pieces = split(" ".join(sentences))
        found += len(pieces)
        total += len(sentences)
        correct += sum(1 for p in pieces if p in sentences)
    return correct / found, correct / total

precision, recall = evaluate(split_sentences)
print(f"preciznost {precision:.1%}   odziv {recall:.1%}")

**Pokušajte:** popravite `split_sentences` u dva koraka i nakon svakog izmjerite preciznost i odziv:

1. rečenica smije počinjati i **znamenkom** ili **navodnikom**: u pogledu unaprijed umjesto `[A-ZČĆĐŠŽ]` napišite `["„]?[A-ZČĆĐŠŽ0-9]`,
2. ne dijelite iza kratice *Dr.*: na početak uzorka dodajte negativni pogled unatrag `(?<!\bDr\.)` i provjerite na primjeru.

In [ ]:
# vaš kod

# 4. Udaljenost uređivanja

S predavanja P4: **minimalna udaljenost uređivanja** je najmanji broj operacija (umetanje, brisanje, zamjena znaka) koje jedan string pretvaraju u drugi. NLTK je računa algoritmom dinamičkog programiranja s predavanja.

In [ ]:
from nltk import edit_distance

print(edit_distance("kuča", "kuća"))                                   # jedna zamjena
print(edit_distance("pjesma", "pisma"))                                # primjer s predavanja
print(edit_distance("intention", "execution"))                         # zamjena košta 1
print(edit_distance("intention", "execution", substitution_cost=2))    # zamjena košta 2 (SLP3)

`edit_distance_align` vraća **poravnanje**: put kroz tablicu D od (0, 0) do (n, m), dobiven povratnim tragom. Usporedite sa zelenim ćelijama na slajdu »Primjer: pjesma → pisma«.

In [ ]:
from nltk import edit_distance_align

edit_distance_align("pjesma", "pisma")

Zamjena dvaju susjednih slova (*ks* → *sk*) česta je tipkarska pogreška, a broji se kao dvije operacije. Parametar `transpositions=True` broji je kao jednu:

In [ ]:
print(edit_distance("hrvatksi", "hrvatski"))
print(edit_distance("hrvatksi", "hrvatski", transpositions=True))

## 4.1 Ispravljanje pogrešaka

Za pogrešno napisanu riječ tražimo **najbližu** riječ u rječniku korpusa. Kad je više riječi jednako blizu, biramo **češću** — zato brojimo koliko se puta svaka riječ pojavljuje.

In [ ]:
from collections import Counter

freq = Counter(t.lower() for _, _, gold in rows for t in gold if t.isalpha())   # riječ -> broj pojavljivanja

def suggest(word, n=5):
    candidates = []
    for w, count in freq.items():
        candidates.append((edit_distance(word, w), -count, w))   # udaljenost, pa češće prije
    candidates.sort()
    return [w for _, _, w in candidates[:n]]

suggest("jezk")   # traje nekoliko sekundi: uspoređujemo s ~29 000 riječi

### Zadatak 3

Ispišite prijedloge za riječi *grda*, *predsjedik*, *sveučilšte* i *hrvatksi*. Zatim u funkciji `suggest` uključite `transpositions=True` i ponovite za *hrvatksi*. Kako se promijenio popis prijedloga i zašto?

In [ ]:
for word in ["grda", "predsjedik", "sveučilšte", "hrvatksi"]:
    print(word, ...)   # dopunite

**Za razmisliti:** za *jezk* drugi prijedlog je *je*, mnogo češća riječ od *jezik*. Je li dovoljno gledati samo udaljenost i učestalost? Što bi još trebalo znati — npr. koje riječi stoje oko pogrešne? (Uskoro: jezični modeli.)